In [ ]:
# Step 1 — Load NMF input data

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "outputs"

nmf_docs = pd.read_csv(
    OUTPUT_DIR / "step9_tfidf_documents.csv"
)

print("Shape:", nmf_docs.shape)

print("\nColumns:")
print(nmf_docs.columns.tolist())

print("\nDocuments by institution category:")
print(
    nmf_docs["category"]
    .value_counts()
    .sort_index()
)

print("\nMissing tfidf_text:")
print(nmf_docs["tfidf_text"].isna().sum())

print("\nEmpty tfidf_text:")
print(
    nmf_docs["tfidf_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

In [ ]:
# Step 2 — Build document-term TF-IDF matrix

from sklearn.feature_extraction.text import TfidfVectorizer

nmf_vectorizer = TfidfVectorizer(
    min_df=3,
    max_df=0.90,
    ngram_range=(1, 1),
    sublinear_tf=True
)

X_nmf = nmf_vectorizer.fit_transform(nmf_docs["tfidf_text"])
feature_names = nmf_vectorizer.get_feature_names_out()

print("Document-term matrix shape:", X_nmf.shape)
print("Number of documents:", X_nmf.shape[0])
print("Number of features:", X_nmf.shape[1])
print("Matrix sparsity:",
      1 - X_nmf.nnz / (X_nmf.shape[0] * X_nmf.shape[1]))

In [ ]:
# Step 3 — Fit candidate NMF models

from sklearn.decomposition import NMF
import pandas as pd

candidate_topics = range(3, 9)

model_results = []
candidate_models = {}

for k in candidate_topics:
    model = NMF(
        n_components=k,
        init="nndsvda",
        random_state=42,
        max_iter=1000
    )

    W = model.fit_transform(X_nmf)

    candidate_models[k] = model

    model_results.append({
        "n_topics": k,
        "reconstruction_error": model.reconstruction_err_,
        "n_iter": model.n_iter_
    })

model_comparison = pd.DataFrame(model_results)

print(model_comparison.to_string(index=False))

In [ ]:
# Inspect top terms for each candidate solution

n_top_terms = 12

for k, model in candidate_models.items():
    print(f"\n{'=' * 70}")
    print(f"{k}-TOPIC SOLUTION")
    print("=" * 70)

    for topic_idx, topic_weights in enumerate(model.components_):
        top_indices = topic_weights.argsort()[-n_top_terms:][::-1]
        top_terms = feature_names[top_indices]

        print(
            f"Topic {topic_idx + 1}: "
            + ", ".join(top_terms)
        )

## Representative documents for candidate solutions

The seven- and eight-topic solutions showed the clearest interpretable structures. 
Representative high-weight documents are therefore inspected to assess whether the 
additional eighth topic captures a substantively distinct pattern or primarily 
fragments an existing topic.

In [ ]:
# Step 4 — Inspect representative documents for k=7 and k=8

for k in [7, 8]:
    model = candidate_models[k]
    W = model.transform(X_nmf)

    print(f"\n{'=' * 90}")
    print(f"{k}-TOPIC SOLUTION — REPRESENTATIVE DOCUMENTS")
    print("=" * 90)

    for topic_idx in range(k):
        top_doc_indices = W[:, topic_idx].argsort()[-5:][::-1]

        print(f"\n--- Topic {topic_idx + 1} ---")

        for rank, doc_idx in enumerate(top_doc_indices, start=1):
            row = nmf_docs.iloc[doc_idx]

            text_preview = str(row["text"]).replace("\n", " ")
            text_preview = text_preview[:500]

            print(
                f"\n{rank}. transcript={row['transcript_id']} | "
                f"category={row['category']} | "
                f"weight={W[doc_idx, topic_idx]:.4f}"
            )
            print(text_preview)

## Diagnose interview-structure artefacts

Some candidate topics appear to reflect repeated demographic and background questions from the interview schedule rather than substantive survivor narratives. These topics are inspected before selecting the final NMF solution.

In [ ]:
# Step 5 — Diagnose interview-structure artefacts

k = 8
model = candidate_models[k]
W = model.transform(X_nmf)

topics_to_check = [1, 5]   # Topic 2 and Topic 6 in zero-based indexing
top_n = 20

for topic_idx in topics_to_check:
    top_doc_indices = W[:, topic_idx].argsort()[-top_n:][::-1]

    print(f"\n{'=' * 90}")
    print(f"TOPIC {topic_idx + 1} — TOP {top_n} DOCUMENTS")
    print("=" * 90)

    topic_categories = []

    for rank, doc_idx in enumerate(top_doc_indices, start=1):
        row = nmf_docs.iloc[doc_idx]
        topic_categories.append(row["category"])

        text_preview = str(row["text"]).replace("\n", " ")
        text_preview = text_preview[:400]

        print(
            f"\n{rank}. transcript={row['transcript_id']} | "
            f"category={row['category']} | "
            f"weight={W[doc_idx, topic_idx]:.4f}"
        )
        print(text_preview)

    print("\nCategory distribution among top documents:")
    print(pd.Series(topic_categories).value_counts())

### Diagnostic interpretation

Inspection of the high-weight documents indicates that Topics 2 and 6 are largely driven by repeated interview-schedule language rather than substantive survivor narratives.

Topic 2 is especially clear: all 20 of its highest-weight documents are classified as `welfare_state`, and most reproduce a recurring sequence of background questions concerning drug or alcohol use, prison or probation experience, housing status, and income. In one representative document, the interviewer explicitly introduces this section as "demographic type questions".

Topic 6 shows a similar pattern. Among its 20 highest-weight documents, 15 are classified as `welfare_state` and 5 as `health`. Many contain standardised questions about gender, sexuality, ethnicity, religion, disability, mental health, housing status, and income.

These patterns suggest that the initial NMF solution is partly capturing the structure of the interview schedule. Because terms such as `housing`, `probation`, `disability`, and `mental health` also occur in the institution dictionary, structured background questions can enter the institution-related corpus even when they do not represent substantive descriptions of institutional experiences.

This diagnostic therefore identifies a corpus-construction artefact that should be addressed before selecting the final number of NMF topics. Importantly, the issue is concentrated in particular passages rather than necessarily affecting entire transcript-category documents; some documents also contain substantive accounts of welfare, health, or social-service interactions. Any subsequent filtering should therefore operate conservatively at passage level rather than removing whole document categories.

In [ ]:
# Step 6 — Load passage-level data and flag likely interview-schedule artefacts

import re
import pandas as pd

passages_nmf = pd.read_csv(
    OUTPUT_DIR / "step9_passages_for_nmf.csv"
)

print("Shape:", passages_nmf.shape)
print(passages_nmf.columns.tolist())

# Conservative indicators of structured demographic/background questioning
artefact_patterns = [
    r"\bhousing status\b",
    r"\bhousehold income\b",
    r"\bindividual income\b",
    r"\bannual income\b",
    r"\bunder £?15[,\s]?000\b",
    r"\b15[,\s]?000.*30[,\s]?000\b",
    r"\bdrug(?:s)? or alcohol\b",
    r"\bprison or probation\b",
    r"\bexperience of prison or probation\b",
    r"\bphysical disability\b",
    r"\bmental health problem\b",
    r"\bidentified ethnicity\b",
    r"\bidentified sexuality\b",
    r"\bdemographic type questions\b",
]

def count_artefact_signals(text):
    text = str(text).lower()
    return sum(
        bool(re.search(pattern, text))
        for pattern in artefact_patterns
    )

passages_nmf["artefact_signal_count"] = (
    passages_nmf["passage_text"]
    .fillna("")
    .apply(count_artefact_signals)
)

# Require multiple signals to avoid deleting genuine substantive mentions
passages_nmf["likely_interview_artefact"] = (
    passages_nmf["artefact_signal_count"] >= 2
)

print("\nLikely artefact passages:")
print(passages_nmf["likely_interview_artefact"].value_counts())

print("\nBy institution category:")
print(
    pd.crosstab(
        passages_nmf["category"],
        passages_nmf["likely_interview_artefact"]
    )
)

## Validation of artefact screening

The conservative screening rule flagged 134 of 4,154 passages (3.2%). Flagged passages were concentrated in `welfare_state` and, to a lesser extent, `health`, while very few passages from the other institution categories were affected.

Before excluding these passages, a sample of flagged and borderline cases is inspected manually to assess whether the rule distinguishes structured interview questions from substantive institutional narratives with acceptable precision.

In [ ]:
# Step 7 — Validate artefact screening

# 1. Random sample of passages flagged as artefacts
flagged_sample = (
    passages_nmf[
        passages_nmf["likely_interview_artefact"]
    ]
    .sample(n=min(20, passages_nmf["likely_interview_artefact"].sum()),
            random_state=42)
)

print("=" * 90)
print("FLAGGED ARTEFACT SAMPLE")
print("=" * 90)

for _, row in flagged_sample.iterrows():
    print(
        f"\ntranscript={row['transcript_id']} | "
        f"category={row['category']} | "
        f"signals={row['artefact_signal_count']}"
    )
    print(str(row["passage_text"])[:600])


# 2. Borderline cases: exactly one artefact signal
borderline = passages_nmf[
    passages_nmf["artefact_signal_count"] == 1
]

print("\n\n" + "=" * 90)
print("BORDERLINE SAMPLE — ONE ARTEFACT SIGNAL")
print("=" * 90)

borderline_sample = borderline.sample(
    n=min(20, len(borderline)),
    random_state=42
)

for _, row in borderline_sample.iterrows():
    print(
        f"\ntranscript={row['transcript_id']} | "
        f"category={row['category']} | "
        f"signals={row['artefact_signal_count']}"
    )
    print(str(row["passage_text"])[:600])

print("\nNumber of borderline passages:", len(borderline))

In [ ]:
# Step 8 — Refine interview-structure artefact detection

import re

# Broad demographic/background domains
domain_patterns = {
    "gender_sexuality": [
        r"\bgender\b",
        r"\bsexuality\b",
        r"\bheterosexual\b",
    ],

    "ethnicity_nationality": [
        r"\bethnicity\b",
        r"\bnationality\b",
        r"\bwhite british\b",
    ],

    "religion_faith": [
        r"\breligion\b",
        r"\bfaith\b",
        r"\breligious\b",
    ],

    "disability": [
        r"\bphysical disabilit",
        r"\bdisabilit",
        r"\blearning (?:need|disabilit)",
    ],

    "mental_health": [
        r"\bmental health\b",
        r"\bdepression\b",
        r"\banxiety\b",
    ],

    "drug_alcohol": [
        r"\bdrug\b",
        r"\balcohol\b",
    ],

    "prison_probation": [
        r"\bprison\b",
        r"\bprobation\b",
    ],

    "housing": [
        r"\bhousing status\b",
        r"\bown(?:er|ing)?\b",
        r"\brent(?:ing|ed)?\b",
        r"\bcouncil tenant\b",
    ],

    "income": [
        r"\bincome\b",
        r"\b£?15[,\s]?000\b",
        r"\b30[,\s]?000\b",
    ],
}

# Very strong explicit indicators
explicit_schedule_patterns = [
    r"\bdemographic questions\b",
    r"\bdemographic type questions\b",
    r"\blast section of questions\b",
    r"\bjust some demographic\b",
]


def detect_domains(text):
    text = str(text).lower()

    matched_domains = []

    for domain, patterns in domain_patterns.items():
        if any(re.search(pattern, text) for pattern in patterns):
            matched_domains.append(domain)

    explicit_schedule = any(
        re.search(pattern, text)
        for pattern in explicit_schedule_patterns
    )

    return matched_domains, explicit_schedule


def refined_artefact_flag(text):
    domains, explicit_schedule = detect_domains(text)

    # Explicit acknowledgement of demographic section
    if explicit_schedule:
        return True

    # Multiple demographic/background domains in one passage
    if len(domains) >= 4:
        return True

    return False


passages_nmf[
    ["matched_domains", "explicit_schedule"]
] = passages_nmf["passage_text"].apply(
    lambda x: pd.Series(detect_domains(x))
)

passages_nmf["n_demographic_domains"] = (
    passages_nmf["matched_domains"].apply(len)
)

passages_nmf["refined_interview_artefact"] = (
    passages_nmf["passage_text"].apply(refined_artefact_flag)
)

print("Refined artefact counts:")
print(
    passages_nmf["refined_interview_artefact"]
    .value_counts()
)

print("\nBy institution category:")
print(
    pd.crosstab(
        passages_nmf["category"],
        passages_nmf["refined_interview_artefact"]
    )
)

print("\nDomain-count distribution:")
print(
    passages_nmf["n_demographic_domains"]
    .value_counts()
    .sort_index()
)

## Validation of the refined screening rule

The refined rule identified 145 passages (3.5% of the passage-level corpus). 
The overall number remained limited, but the distribution shifted from the initial 
screening, particularly between `welfare_state` and `health`. Passages whose 
classification changed between the two rules are therefore inspected directly 
before the refined rule is adopted for filtering.

In [ ]:
# Step 9 — Compare initial and refined artefact classifications

# Newly detected by the refined rule
newly_flagged = passages_nmf[
    (~passages_nmf["likely_interview_artefact"]) &
    (passages_nmf["refined_interview_artefact"])
].copy()

# Previously flagged but no longer detected
no_longer_flagged = passages_nmf[
    (passages_nmf["likely_interview_artefact"]) &
    (~passages_nmf["refined_interview_artefact"])
].copy()

print("Newly flagged by refined rule:", len(newly_flagged))
print("No longer flagged by refined rule:", len(no_longer_flagged))

print("\nNewly flagged by category:")
print(newly_flagged["category"].value_counts())

print("\nNo longer flagged by category:")
print(no_longer_flagged["category"].value_counts())


def print_changed_cases(df, title, n=20):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    sample = df.sample(
        n=min(n, len(df)),
        random_state=42
    )

    for _, row in sample.iterrows():
        print(
            f"\ntranscript={row['transcript_id']} | "
            f"category={row['category']} | "
            f"domains={row['matched_domains']} | "
            f"n_domains={row['n_demographic_domains']} | "
            f"explicit={row['explicit_schedule']}"
        )
        print(str(row["passage_text"])[:700])


print_changed_cases(
    newly_flagged,
    "NEWLY FLAGGED BY REFINED RULE"
)

print_changed_cases(
    no_longer_flagged,
    "NO LONGER FLAGGED BY REFINED RULE"
)

### Final conservative screening rule

Manual comparison of the initial and refined screening rules showed that no single indicator was sufficient for reliable passage-level exclusion.

The screening criteria were also checked against the original interview schedule, which contains a dedicated demographic and background section covering gender and sexuality, ethnicity and nationality, religion, disability, mental health, drug/alcohol use, prison/probation experience, housing status, income, and financial impact. This provided an external basis for distinguishing recurrent interviewer-generated questionnaire structures from substantive survivor narratives.

The final rule therefore targets two recurrent interview-schedule structures:

1. a demographic identity/background block containing multiple identity-related domains; or
2. a repeated socioeconomic block combining prison/probation, housing, and income questions.

Explicit references to "demographic questions" are retained as diagnostic information but are not used alone to exclude a passage, because reconstructed passages may span the boundary between substantive institutional discussion and the demographic section.

This conservative approach prioritises removal of clearly interviewer-generated structures while retaining passages that may contain substantive survivor accounts.

In [ ]:
# Step 10 — Finalise conservative artefact screening rule

# Identity/background domains that commonly occur together
identity_background_domains = {
    "gender_sexuality",
    "ethnicity_nationality",
    "religion_faith",
    "disability",
    "mental_health",
    "drug_alcohol",
}

def final_artefact_rule(row):
    domains = set(row["matched_domains"])

    # Pattern A:
    # repeated demographic / identity-background questionnaire block
    n_identity_domains = len(
        domains.intersection(identity_background_domains)
    )

    identity_block = n_identity_domains >= 4

    # Pattern B:
    # repeated socioeconomic/background questionnaire sequence
    socioeconomic_block = {
        "prison_probation",
        "housing",
        "income"
    }.issubset(domains)

    return identity_block or socioeconomic_block


passages_nmf["final_interview_artefact"] = (
    passages_nmf.apply(final_artefact_rule, axis=1)
)

# Diagnostic variables for transparency
passages_nmf["n_identity_background_domains"] = (
    passages_nmf["matched_domains"].apply(
        lambda domains: len(
            set(domains).intersection(identity_background_domains)
        )
    )
)

print("Final artefact counts:")
print(
    passages_nmf["final_interview_artefact"]
    .value_counts()
)

print("\nBy institution category:")
print(
    pd.crosstab(
        passages_nmf["category"],
        passages_nmf["final_interview_artefact"]
    )
)

print("\nComparison with previous rules:")
comparison = pd.DataFrame({
    "initial_rule": passages_nmf["likely_interview_artefact"],
    "refined_rule": passages_nmf["refined_interview_artefact"],
    "final_rule": passages_nmf["final_interview_artefact"]
})

print(comparison.sum())

print("\nFinal flagged percentage:")
print(
    round(
        passages_nmf["final_interview_artefact"].mean() * 100,
        2
    ),
    "%"
)

### Interim conservative screening decision

The interim conservative rule identified 121 of 4,154 passages (2.9%) as clear interview-schedule artefacts. These passages were concentrated in `welfare_state` (65 passages) and `health` (54 passages), with only two identified in `support_sector` and none in `police` or `legal`.

This rule was deliberately more conservative than the earlier screening variants. It targeted recurrent demographic identity/background blocks and the repeated prison/probation–housing–income sequence, while avoiding exclusion based solely on explicit references to demographic questioning. The aim was to reduce the influence of interviewer-generated structure while minimising the risk of removing substantive survivor narratives.

This rule was provisionally retained to construct a first cleaned corpus for diagnostic NMF modelling. The resulting topic structure was subsequently inspected for residual interview-schedule effects before the artefact-screening criteria were finalised.

## Construction of the cleaned NMF corpus

Following the diagnostic and validation stages, passages identified by the final conservative rule as clear interview-schedule artefacts were excluded from the NMF input.

Filtering is performed at passage level rather than document or transcript level. This allows structured interviewer-generated material to be removed while retaining other substantive institution-related passages from the same transcript and institution category.

The remaining passages are then re-aggregated into transcript × institution-category documents for the final NMF analysis.

In [ ]:
# Step 11 — Construct cleaned NMF corpus

# Remove passages identified as clear interview-schedule artefacts
clean_passages = passages_nmf[
    ~passages_nmf["final_interview_artefact"]
].copy()

print("Original passages:", len(passages_nmf))
print(
    "Removed artefact passages:",
    passages_nmf["final_interview_artefact"].sum()
)
print("Remaining passages:", len(clean_passages))


# Reconstruct transcript × institution-category documents
clean_docs = (
    clean_passages
    .sort_values(
        ["transcript_id", "category", "passage_start"]
    )
    .groupby(
        ["transcript_id", "category"],
        as_index=False
    )
    .agg(
        text=("passage_text", " ".join),
        n_passages=("passage_text", "size"),
        n_kwic_windows=("n_windows", "sum")
    )
)

clean_docs["char_count"] = clean_docs["text"].str.len()


print("\nClean document shape:")
print(clean_docs.shape)

print("\nDocuments by category:")
print(
    clean_docs["category"]
    .value_counts()
    .sort_index()
)

print("\nOriginal document count:", len(nmf_docs))
print("Clean document count:", len(clean_docs))
print(
    "Documents completely removed:",
    len(nmf_docs) - len(clean_docs)
)

### Interim clean corpus check

After passage-level artefact removal, 4,033 of the original 4,154 passages were retained. Re-aggregation produced 596 transcript × institution-category documents, compared with 627 before filtering.

Thirty-one documents were completely removed: 24 from `welfare_state` and 7 from `health`. No `police`, `legal`, or `support_sector` documents were lost. This pattern is consistent with the earlier diagnostics, which showed that structured demographic and background questions disproportionately affected welfare- and health-related keyword retrieval.

The resulting 596-document corpus was retained as the input for a diagnostic NMF modelling stage. The resulting topic structure was subsequently inspected for residual interview-schedule artefacts before the final analytical corpus was fixed.


## Preprocessing of the interim cleaned NMF corpus

The cleaned documents were processed using the same linguistic preprocessing pipeline established in the TF-IDF analysis. This maintains consistency between the lexical comparison and NMF stages, while ensuring that changes in the final topic structure can be attributed primarily to the removal of interview-schedule artefacts rather than to changes in text preprocessing.

In [ ]:
# Step 12 — Apply the established preprocessing pipeline to clean documents

import spacy
from tqdm.auto import tqdm

nlp = spacy.load("en_core_web_sm")

KEEP_POS = {"NOUN", "VERB", "ADJ", "ADV"}

CUSTOM_STOPWORDS = {
    "interviewer",
    "respondent",
    "know",
    "yeah",
    "yes",
    "okay",
    "ok",
    "actually",
    "kind",
    "sort",
    "mean",
    "thing",
    "lot",
    "inaudible"
}


def preprocess_text(text):
    doc = nlp(str(text))

    tokens = []

    for token in doc:
        if not token.is_alpha:
            continue

        if token.is_stop:
            continue

        if token.pos_ not in KEEP_POS:
            continue

        lemma = token.lemma_.lower().strip()

        if len(lemma) < 2:
            continue

        tokens.append(lemma)

    return " ".join(tokens)


def remove_custom_stopwords(text):
    tokens = text.split()

    tokens = [
        token for token in tokens
        if token not in CUSTOM_STOPWORDS
    ]

    return " ".join(tokens)


tqdm.pandas()

clean_docs["processed_text"] = (
    clean_docs["text"]
    .fillna("")
    .progress_apply(preprocess_text)
)

clean_docs["tfidf_text"] = (
    clean_docs["processed_text"]
    .fillna("")
    .apply(remove_custom_stopwords)
)

clean_docs["processed_word_count"] = (
    clean_docs["processed_text"]
    .str.split()
    .str.len()
)

clean_docs["tfidf_word_count"] = (
    clean_docs["tfidf_text"]
    .str.split()
    .str.len()
)


print("Documents processed:", len(clean_docs))

print("\nTF-IDF input word-count distribution:")
print(
    clean_docs["tfidf_word_count"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
)

print(
    "\nEmpty documents after preprocessing:",
    (clean_docs["tfidf_word_count"] == 0).sum()
)

In [ ]:
# Step 13 — Build cleaned document-term TF-IDF matrix

from sklearn.feature_extraction.text import TfidfVectorizer

clean_vectorizer = TfidfVectorizer(
    min_df=3,
    max_df=0.90,
    ngram_range=(1, 1),
    sublinear_tf=True
)

X_clean = clean_vectorizer.fit_transform(
    clean_docs["tfidf_text"]
)

clean_feature_names = (
    clean_vectorizer.get_feature_names_out()
)

print("Clean document-term matrix shape:", X_clean.shape)
print("Number of documents:", X_clean.shape[0])
print("Number of features:", X_clean.shape[1])

print(
    "Matrix sparsity:",
    1 - X_clean.nnz /
    (X_clean.shape[0] * X_clean.shape[1])
)

Candidate NMF models with 3–8 topics were fitted to the cleaned TF-IDF matrix. 
The same range of topic numbers used in the initial diagnostic modelling was retained 
to allow direct comparison before and after artefact removal.

Model assessment considers both reconstruction error and substantive interpretability. 
Particular attention is given to whether the interview-schedule structures identified 
in the initial models persist after cleaning.

In [ ]:
# Step 14 — Fit candidate NMF models on the cleaned corpus

from sklearn.decomposition import NMF
import pandas as pd

candidate_models_clean = {}
candidate_results_clean = []

for n_topics in range(3, 9):

    model = NMF(
        n_components=n_topics,
        init="nndsvda",
        random_state=42,
        max_iter=1000
    )

    W = model.fit_transform(X_clean)
    H = model.components_

    candidate_models_clean[n_topics] = {
        "model": model,
        "W": W,
        "H": H
    }

    candidate_results_clean.append({
        "n_topics": n_topics,
        "reconstruction_error": model.reconstruction_err_,
        "n_iter": model.n_iter_
    })


candidate_results_clean = pd.DataFrame(
    candidate_results_clean
)

print(candidate_results_clean.to_string(index=False))

In [ ]:
# Step 15 — Inspect top terms across candidate NMF solutions

def print_top_terms(model, feature_names, n_top_words=12):

    for topic_idx, topic in enumerate(model.components_):

        top_indices = topic.argsort()[
            :-n_top_words - 1:-1
        ]

        top_terms = [
            feature_names[i]
            for i in top_indices
        ]

        print(
            f"Topic {topic_idx + 1}: "
            + ", ".join(top_terms)
        )


for n_topics in range(3, 9):

    print("\n" + "=" * 75)
    print(f"{n_topics}-TOPIC CLEAN SOLUTION")
    print("=" * 75)

    print_top_terms(
        candidate_models_clean[n_topics]["model"],
        clean_feature_names,
        n_top_words=12
    )

### Interpretation of the cleaned candidate models

Removal of clear interview-schedule artefacts changed the latent structure of the NMF solutions. The previously prominent prison/probation–drug/alcohol questionnaire pattern no longer emerged as an independent topic.

Across the higher-dimensional solutions, several patterns became increasingly stable, including police intervention, health and counselling, specialist support, financial and housing-related experiences, and broader justice narratives. The eight-topic solution additionally separated a distinct legal-process pattern and a social-services-related pattern.

Before selecting the final model, representative documents from the eight-topic solution are inspected to determine whether each topic reflects a substantively interpretable narrative structure. Particular attention is given to Topic 2, which retains housing, income, disability and health-related terms and may contain residual interview-structure effects.

In [ ]:
# Step 16 — Inspect representative documents for the clean 8-topic solution

k = 8

model_8 = candidate_models_clean[k]["model"]
W_8 = candidate_models_clean[k]["W"]

top_n_docs = 5

for topic_idx in range(k):

    top_doc_indices = (
        W_8[:, topic_idx]
        .argsort()[-top_n_docs:][::-1]
    )

    print("\n" + "=" * 90)
    print(f"CLEAN TOPIC {topic_idx + 1}")
    print("=" * 90)

    for rank, doc_idx in enumerate(
        top_doc_indices,
        start=1
    ):
        row = clean_docs.iloc[doc_idx]

        text_preview = (
            str(row["text"])
            .replace("\n", " ")
            [:700]
        )

        print(
            f"\n{rank}. "
            f"transcript={row['transcript_id']} | "
            f"category={row['category']} | "
            f"weight={W_8[doc_idx, topic_idx]:.4f}"
        )

        print(text_preview)

In [ ]:
# Step 17 — Trace residual NMF artefacts back to source passages

diagnostic_cases = [
    ("JUST053", "welfare_state"),
    ("JUST033", "welfare_state"),
    ("JUST038", "welfare_state"),
    ("JUST269", "welfare_state"),
    ("JUST182", "welfare_state"),
    ("JUST317", "welfare_state")
]

for transcript_id, category in diagnostic_cases:

    subset = passages_nmf[
        (passages_nmf["transcript_id"] == transcript_id) &
        (passages_nmf["category"] == category)
    ].sort_values("passage_start")

    print("\n" + "=" * 90)
    print(transcript_id, "|", category)
    print("=" * 90)

    for _, row in subset.iterrows():

        print(
            f"\npassage_start={row['passage_start']} | "
            f"final_artefact={row['final_interview_artefact']}"
        )

        print(
            str(row["passage_text"])
            .replace("\n", " ")[:1000]
        )

In [ ]:
# Step 18 — Add final residual demographic-block criterion

identity_background_domains = {
    "gender_sexuality",
    "ethnicity_nationality",
    "religion_faith",
    "disability",
    "mental_health",
    "drug_alcohol",
}

def final_artefact_rule_v2(row):
    domains = set(row["matched_domains"])

    n_identity_domains = len(
        domains.intersection(identity_background_domains)
    )

    # Existing rule A
    identity_block = n_identity_domains >= 4

    # Existing rule B
    socioeconomic_block = {
        "prison_probation",
        "housing",
        "income"
    }.issubset(domains)

    # Final residual demographic pattern:
    # housing + income combined with multiple identity/background questions
    residual_demographic_block = (
        {"housing", "income"}.issubset(domains)
        and n_identity_domains >= 2
    )

    return (
        identity_block
        or socioeconomic_block
        or residual_demographic_block
    )


passages_nmf["final_interview_artefact_v2"] = (
    passages_nmf.apply(
        final_artefact_rule_v2,
        axis=1
    )
)

print("Previous final rule:")
print(passages_nmf["final_interview_artefact"].value_counts())

print("\nUpdated final rule:")
print(passages_nmf["final_interview_artefact_v2"].value_counts())

print("\nAdditional passages flagged:")
additional = (
    passages_nmf["final_interview_artefact_v2"]
    & ~passages_nmf["final_interview_artefact"]
)

print(additional.sum())

print("\nAdditional passages by category:")
print(
    passages_nmf.loc[
        additional, "category"
    ].value_counts()
)

### Final refinement of artefact screening

Inspection of representative documents from the cleaned NMF model revealed a small number of residual demographic questionnaire passages. These typically combined housing and household-income questions with multiple identity or health-related demographic questions.

A final conservative criterion was therefore added to capture this specific recurrent structure. This identified 14 additional passages: 11 in `welfare_state` and 3 in `health`. The final screening rule consequently excluded 135 of 4,154 passages (3.25%).

No additional passages were identified in `police`, `legal`, or `support_sector`. The screening criteria were fixed at this point and no further rule refinement was undertaken.

In [ ]:
# Step 19 — Construct final cleaned NMF corpus

final_clean_passages = passages_nmf[
    ~passages_nmf["final_interview_artefact_v2"]
].copy()

final_clean_docs = (
    final_clean_passages
    .sort_values(
        ["transcript_id", "category", "passage_start"]
    )
    .groupby(
        ["transcript_id", "category"],
        as_index=False
    )
    .agg(
        text=("passage_text", " ".join),
        n_passages=("passage_text", "size"),
        n_kwic_windows=("n_windows", "sum")
    )
)

final_clean_docs["char_count"] = (
    final_clean_docs["text"].str.len()
)

print("Original passages:", len(passages_nmf))
print(
    "Final removed passages:",
    passages_nmf["final_interview_artefact_v2"].sum()
)
print("Remaining passages:", len(final_clean_passages))

print("\nFinal clean documents:", len(final_clean_docs))

print("\nDocuments by category:")
print(
    final_clean_docs["category"]
    .value_counts()
    .sort_index()
)

print(
    "\nDocuments completely removed:",
    len(nmf_docs) - len(final_clean_docs)
)

In [ ]:
# Step 20 — Preprocess final cleaned NMF documents

tqdm.pandas()

final_clean_docs["processed_text"] = (
    final_clean_docs["text"]
    .fillna("")
    .progress_apply(preprocess_text)
)

final_clean_docs["tfidf_text"] = (
    final_clean_docs["processed_text"]
    .fillna("")
    .apply(remove_custom_stopwords)
)

final_clean_docs["processed_word_count"] = (
    final_clean_docs["processed_text"]
    .str.split()
    .str.len()
)

final_clean_docs["tfidf_word_count"] = (
    final_clean_docs["tfidf_text"]
    .str.split()
    .str.len()
)

print("Documents processed:", len(final_clean_docs))

print(
    "Empty documents after preprocessing:",
    (final_clean_docs["tfidf_word_count"] == 0).sum()
)

print("\nTF-IDF word-count summary:")
print(final_clean_docs["tfidf_word_count"].describe())

## Final TF-IDF representation

The final cleaned corpus was converted into a shared TF-IDF document-term matrix using the same vectorisation settings as the earlier modelling stages. All 588 documents remained non-empty after preprocessing.

In [ ]:
# Step 21 — Build final TF-IDF matrix

final_vectorizer = TfidfVectorizer(
    min_df=3,
    max_df=0.90,
    ngram_range=(1, 1),
    sublinear_tf=True
)

X_final = final_vectorizer.fit_transform(
    final_clean_docs["tfidf_text"]
)

final_feature_names = (
    final_vectorizer.get_feature_names_out()
)

print("Final document-term matrix shape:", X_final.shape)
print("Number of documents:", X_final.shape[0])
print("Number of features:", X_final.shape[1])

print(
    "Matrix sparsity:",
    1 - X_final.nnz /
    (X_final.shape[0] * X_final.shape[1])
)

## Final candidate NMF models

NMF models with three to eight topics were fitted to the final cleaned TF-IDF matrix. This constitutes the final modelling stage: the corpus and artefact-screening criteria were fixed before these models were evaluated, and no further corpus modification was undertaken on the basis of the resulting topic structure.

Candidate solutions are compared using reconstruction error together with topic interpretability and separation. The final number of topics is selected on substantive rather than reconstruction-error grounds alone.

In [ ]:
# Step 22 — Fit final candidate NMF models

from sklearn.decomposition import NMF
import pandas as pd

final_candidate_models = {}
final_candidate_results = []

for n_topics in range(3, 9):

    model = NMF(
        n_components=n_topics,
        init="nndsvda",
        random_state=42,
        max_iter=1000
    )

    W = model.fit_transform(X_final)
    H = model.components_

    final_candidate_models[n_topics] = {
        "model": model,
        "W": W,
        "H": H
    }

    final_candidate_results.append({
        "n_topics": n_topics,
        "reconstruction_error": model.reconstruction_err_,
        "n_iter": model.n_iter_
    })


final_candidate_results = pd.DataFrame(
    final_candidate_results
)

print(final_candidate_results.to_string(index=False))

In [ ]:
# Step 23 — Inspect top terms across final candidate solutions

for n_topics in range(3, 9):

    print("\n" + "=" * 75)
    print(f"{n_topics}-TOPIC FINAL SOLUTION")
    print("=" * 75)

    print_top_terms(
        final_candidate_models[n_topics]["model"],
        final_feature_names,
        n_top_words=12
    )

### Candidate-model interpretation

The final cleaned models showed substantially clearer topic separation than the initial diagnostic models, with no independent topic dominated by demographic questionnaire structure.

The eight-topic solution provided the most substantively differentiated structure. In particular, it separated civil/family legal processes (e.g. solicitors, legal aid, divorce and mediation) from criminal proceedings (e.g. trial, sentence, plea, evidence and statements), while retaining stable topics relating to police involvement, health, material consequences, specialist support and social services.

The eight-topic model was therefore treated as the leading candidate. Representative documents were inspected before final selection, particularly for the broader justice-related Topic 5 and the newly separated criminal-process Topic 8.

In [ ]:
# Step 24 — Inspect representative documents for final k=8 candidate

final_k = 8

final_model_8 = final_candidate_models[final_k]["model"]
final_W_8 = final_candidate_models[final_k]["W"]

for topic_idx in range(final_k):

    top_doc_indices = (
        final_W_8[:, topic_idx]
        .argsort()[-5:][::-1]
    )

    print("\n" + "=" * 90)
    print(f"FINAL CANDIDATE TOPIC {topic_idx + 1}")
    print("=" * 90)

    for rank, doc_idx in enumerate(top_doc_indices, start=1):

        row = final_clean_docs.iloc[doc_idx]

        text_preview = (
            str(row["text"])
            .replace("\n", " ")
            [:700]
        )

        print(
            f"\n{rank}. "
            f"transcript={row['transcript_id']} | "
            f"category={row['category']} | "
            f"weight={final_W_8[doc_idx, topic_idx]:.4f}"
        )

        print(text_preview)

### Final topic-number decision

The eight-topic solution was selected as the final NMF model. Reconstruction error decreased progressively as the number of topics increased, but this metric was not used as the sole selection criterion because lower reconstruction error is expected as model complexity increases.

The final decision was based primarily on substantive interpretability, topic separation, and inspection of the highest-weight documents for each candidate topic. The eight-topic solution produced distinguishable patterns relating to police intervention, health and mental-health support, housing and financial consequences, specialist refuge/support services, justice perceptions and institutional engagement, civil/family legal processes, social-service intervention, and criminal justice proceedings.

The distinction between Topics 6 and 8 was particularly informative. Topic 6 was associated primarily with solicitors, legal aid, divorce, mediation and family-related legal processes, whereas Topic 8 captured criminal proceedings including evidence, trial, prosecution and sentencing. Inspection of representative documents indicated that this distinction reflected substantively different institutional processes rather than a trivial subdivision of an existing topic.

Some residual structured interview language remained within the housing and financial topic. This topic is therefore interpreted cautiously and is not treated as direct evidence of survivors' evaluations of welfare institutions.

Final model: k = 8.

In [ ]:
# Step 25 — Extract final NMF topic and document weights

final_k = 8

final_model = final_candidate_models[final_k]["model"]
final_W = final_candidate_models[final_k]["W"]
final_H = final_candidate_models[final_k]["H"]

# -----------------------------
# 1. Topic-term weights
# -----------------------------
topic_term_rows = []

for topic_idx, topic_weights in enumerate(final_H):

    top_indices = topic_weights.argsort()[::-1]

    for rank, term_idx in enumerate(top_indices[:20], start=1):

        topic_term_rows.append({
            "topic": topic_idx + 1,
            "rank": rank,
            "term": final_feature_names[term_idx],
            "weight": topic_weights[term_idx]
        })

final_topic_terms = pd.DataFrame(topic_term_rows)


# -----------------------------
# 2. Document-topic weights
# -----------------------------
topic_columns = [
    f"topic_{i}"
    for i in range(1, final_k + 1)
]

final_document_topics = pd.DataFrame(
    final_W,
    columns=topic_columns
)

final_document_topics.insert(
    0,
    "category",
    final_clean_docs["category"].values
)

final_document_topics.insert(
    0,
    "transcript_id",
    final_clean_docs["transcript_id"].values
)

final_document_topics["dominant_topic"] = (
    final_W.argmax(axis=1) + 1
)

final_document_topics["dominant_topic_weight"] = (
    final_W.max(axis=1)
)


print("Final topic-term table:")
print(final_topic_terms.head(20))

print("\nDocument-topic matrix shape:")
print(final_document_topics.shape)

print("\nDominant topic distribution:")
print(
    final_document_topics["dominant_topic"]
    .value_counts()
    .sort_index()
)

## Topic distribution across institution categories

To examine how latent narrative patterns varied across institution-related retrieval categories, document-topic weights from the final NMF model were aggregated by category.

Here, `category` refers to the institution category assigned during the dictionary-based KWIC retrieval and corpus-construction stages (`police`, `legal`, `health`, `support_sector`, and `welfare_state`). It should therefore be interpreted as a retrieval-based analytical grouping rather than as a definitive manual classification of the substantive content of each document. Documents may contain references to institutions or experiences beyond the category through which they were retrieved.

Because NMF represents each document as a weighted combination of topics, mean topic weights are used as the primary comparison rather than assigning each document exclusively to its dominant topic. Dominant-topic proportions are retained as a complementary descriptive measure.

In [ ]:
# Step 26 — Compare mean topic weights across institution categories

topic_columns = [
    f"topic_{i}"
    for i in range(1, 9)
]

mean_topic_by_category = (
    final_document_topics
    .groupby("category")[topic_columns]
    .mean()
)

print("Mean topic weights by institution category:")
print(
    mean_topic_by_category
    .round(4)
    .to_string()
)

In [ ]:
# Step 27 — Compare dominant-topic proportions across institution categories

dominant_topic_proportions = (
    pd.crosstab(
        final_document_topics["category"],
        final_document_topics["dominant_topic"],
        normalize="index"
    )
    .reindex(columns=range(1, 9), fill_value=0)
)

dominant_topic_proportions.columns = [
    f"topic_{i}"
    for i in range(1, 9)
]

print("Dominant-topic proportions by institution category:")
print(
    dominant_topic_proportions
    .round(3)
    .to_string()
)

### Interpretation of cross-category topic distributions

The final topic distributions showed clear differences across institution categories. Health-category documents were the most concentrated, with Topic 2 (health and mental-health support) dominant in 73.4% of documents. Police-category documents were divided primarily between direct police intervention (Topic 1, 44.6%) and broader justice perceptions and institutional engagement (Topic 5, 38.1%).

Legal-category documents displayed a more differentiated structure. Topic 6, relating to legal and family/civil processes, was dominant in 47.4% of legal documents, while Topic 5 (justice and institutional engagement) and Topic 8 (criminal proceedings) accounted for 25.6% and 24.1% respectively. This supports the distinction between different forms of legal experience identified in the eight-topic solution.

Support-sector documents were primarily associated with refuge and specialist support (Topic 4, 41.9%), but justice/institutional-engagement narratives were also prominent (24.7%). Welfare/state documents showed a different pattern, divided almost equally between housing/financial issues (Topic 3) and social-service intervention (Topic 7), each dominant in 38.9% of documents. Topic 3 is interpreted cautiously because representative-document inspection identified some remaining influence from structured financial questions in the interview schedule.

Overall, the results indicate that the retrieval categories differ not only in their relatively distinctive vocabulary but also in their document-level latent textual structures.

In [ ]:
# Step 28 — Visualise mean topic weights across institution categories

import matplotlib.pyplot as plt

topic_labels = {
    "topic_1": "Police contact / intervention",
    "topic_2": "Health & mental-health support",
    "topic_3": "Housing & financial consequences",
    "topic_4": "Refuge & specialist support",
    "topic_5": "Justice perceptions & experiences",
    "topic_6": "Civil / family legal processes",
    "topic_7": "Social services intervention",
    "topic_8": "Criminal justice proceedings"
}

heatmap_data = mean_topic_by_category.rename(
    columns=topic_labels
)

fig, ax = plt.subplots(figsize=(14, 5.5))

im = ax.imshow(
    heatmap_data.values,
    aspect="auto"
)

ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(
    heatmap_data.columns,
    rotation=40,
    ha="right"
)

ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(
    heatmap_data.index
)

ax.set_xlabel("NMF topic")
ax.set_ylabel("Institution category")
ax.set_title(
    "Mean NMF Topic Weights across Institution Categories"
)

# Add values inside cells
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        ax.text(
            j,
            i,
            f"{heatmap_data.iloc[i, j]:.3f}",
            ha="center",
            va="center"
        )

fig.colorbar(
    im,
    ax=ax,
    label="Mean topic weight"
)

plt.tight_layout()
plt.show()

In [ ]:
# Step 29 — Export final NMF results

# 1. Topic-term weights
final_topic_terms.to_csv(
    OUTPUT_DIR / "step10_nmf_final_topic_terms.csv",
    index=False
)

# 2. Document-topic weights
final_document_topics.to_csv(
    OUTPUT_DIR / "step10_nmf_document_topic_weights.csv",
    index=False
)

# 3. Mean topic weights by institution category
mean_topic_by_category.to_csv(
    OUTPUT_DIR / "step10_nmf_mean_topic_weights_by_category.csv"
)

# 4. Dominant-topic proportions by institution category
dominant_topic_proportions.to_csv(
    OUTPUT_DIR / "step10_nmf_dominant_topic_proportions.csv"
)

print("Final NMF tables saved:")
print("- step10_nmf_final_topic_terms.csv")
print("- step10_nmf_document_topic_weights.csv")
print("- step10_nmf_mean_topic_weights_by_category.csv")
print("- step10_nmf_dominant_topic_proportions.csv")
print(f"\nOutput directory: {OUTPUT_DIR}")

In [ ]:
# Step 30 — Build final NMF topic summary table

topic_names = {
    1: "Police contact and intervention",
    2: "Health and mental-health support",
    3: "Housing and financial consequences",
    4: "Refuge and specialist support",
    5: "Justice perceptions and institutional engagement",
    6: "Civil and family legal processes",
    7: "Social services intervention",
    8: "Criminal justice proceedings"
}

summary_rows = []

for topic_num in range(1, 9):

    # Top 10 terms
    top_terms = (
        final_topic_terms[
            final_topic_terms["topic"] == topic_num
        ]
        .sort_values("rank")
        .head(10)["term"]
        .tolist()
    )

    # Number of documents where this is the dominant topic
    dominant_n = (
        final_document_topics["dominant_topic"]
        .eq(topic_num)
        .sum()
    )

    dominant_pct = (
        dominant_n / len(final_document_topics) * 100
    )

    # Institution category with highest mean topic weight
    topic_col = f"topic_{topic_num}"

    strongest_category = (
        mean_topic_by_category[topic_col]
        .idxmax()
    )

    strongest_mean_weight = (
        mean_topic_by_category.loc[
            strongest_category,
            topic_col
        ]
    )

    summary_rows.append({
        "topic": topic_num,
        "topic_label": topic_names[topic_num],
        "top_10_terms": ", ".join(top_terms),
        "dominant_documents_n": dominant_n,
        "dominant_documents_pct": dominant_pct,
        "strongest_institution_category": strongest_category,
        "mean_weight_in_strongest_category": strongest_mean_weight
    })


final_topic_summary = pd.DataFrame(summary_rows)

# Display with readable formatting
display_summary = final_topic_summary.copy()

display_summary["dominant_documents_pct"] = (
    display_summary["dominant_documents_pct"]
    .round(1)
)

display_summary["mean_weight_in_strongest_category"] = (
    display_summary["mean_weight_in_strongest_category"]
    .round(3)
)

display(display_summary)

In [ ]:
# Step 31 — Save final NMF topic summary

final_topic_summary.to_csv(
    OUTPUT_DIR / "step10_nmf_final_topic_summary.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "step10_nmf_final_topic_summary.csv"
)

In [ ]:
# Step 32 — Create and save publication-ready NMF heatmap

import matplotlib.pyplot as plt

topic_labels_short = {
    "topic_1": "Police contact\n& intervention",
    "topic_2": "Health & mental-\nhealth support",
    "topic_3": "Housing & financial\nconsequences",
    "topic_4": "Refuge & specialist\nsupport",
    "topic_5": "Justice perceptions\n& engagement",
    "topic_6": "Civil & family\nlegal processes",
    "topic_7": "Social services\nintervention",
    "topic_8": "Criminal justice\nproceedings"
}

category_labels = {
    "health": "Health",
    "legal": "Legal",
    "police": "Police",
    "support_sector": "Support sector",
    "welfare_state": "Welfare / state"
}

plot_data = (
    mean_topic_by_category
    .rename(columns=topic_labels_short)
    .rename(index=category_labels)
)

fig, ax = plt.subplots(figsize=(14, 5.8))

im = ax.imshow(
    plot_data.values,
    aspect="auto"
)

ax.set_xticks(range(len(plot_data.columns)))
ax.set_xticklabels(
    plot_data.columns,
    rotation=35,
    ha="right"
)

ax.set_yticks(range(len(plot_data.index)))
ax.set_yticklabels(plot_data.index)

ax.set_xlabel("NMF topic")
ax.set_ylabel("Institution category")
ax.set_title(
    "Mean NMF Topic Weights across Institution Categories"
)

for i in range(plot_data.shape[0]):
    for j in range(plot_data.shape[1]):
        ax.text(
            j,
            i,
            f"{plot_data.iloc[i, j]:.3f}",
            ha="center",
            va="center"
        )

fig.colorbar(
    im,
    ax=ax,
    label="Mean topic weight"
)

plt.tight_layout()

figure_path = (
    OUTPUT_DIR /
    "step10_nmf_topic_weights_heatmap.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", figure_path)

In [ ]:
# Step 33 — Extract representative documents for each final topic

representative_rows = []

n_representative = 10

for topic_num in range(1, 9):

    topic_col = f"topic_{topic_num}"

    top_indices = (
        final_document_topics[topic_col]
        .nlargest(n_representative)
        .index
    )

    for rank, doc_idx in enumerate(top_indices, start=1):

        doc = final_clean_docs.loc[doc_idx]

        representative_rows.append({
            "topic": topic_num,
            "topic_label": topic_names[topic_num],
            "rank": rank,
            "transcript_id": doc["transcript_id"],
            "category": doc["category"],
            "topic_weight": final_document_topics.loc[
                doc_idx, topic_col
            ],
            "text": doc["text"]
        })


final_representative_documents = pd.DataFrame(
    representative_rows
)

print("Representative documents:", len(final_representative_documents))

print("\nDocuments per topic:")
print(
    final_representative_documents
    .groupby("topic")
    .size()
)

print("\nInstitution categories within representative documents:")
print(
    pd.crosstab(
        final_representative_documents["topic"],
        final_representative_documents["category"]
    )
)

In [ ]:
# Step 34 — Save final representative documents

final_representative_documents.to_csv(
    OUTPUT_DIR / "step10_nmf_representative_documents.csv",
    index=False
)

print(
    "Saved:",
    OUTPUT_DIR / "step10_nmf_representative_documents.csv"
)

In [ ]:
# Step 35 — Audit final NMF outputs

final_nmf_outputs = {
    "Final topic terms":
        OUTPUT_DIR / "step10_nmf_final_topic_terms.csv",

    "Document-topic weights":
        OUTPUT_DIR / "step10_nmf_document_topic_weights.csv",

    "Mean topic weights by category":
        OUTPUT_DIR / "step10_nmf_mean_topic_weights_by_category.csv",

    "Dominant-topic proportions":
        OUTPUT_DIR / "step10_nmf_dominant_topic_proportions.csv",

    "Final topic summary":
        OUTPUT_DIR / "step10_nmf_final_topic_summary.csv",

    "Representative documents":
        OUTPUT_DIR / "step10_nmf_representative_documents.csv",

    "Topic-weight heatmap":
        OUTPUT_DIR / "step10_nmf_topic_weights_heatmap.png"
}

print("FINAL NMF OUTPUT AUDIT")
print("=" * 70)

all_present = True

for name, path in final_nmf_outputs.items():

    exists = path.exists()

    print(
        f"{'OK' if exists else 'MISSING':8} "
        f"{name:35} {path.name}"
    )

    if not exists:
        all_present = False

print("=" * 70)

if all_present:
    print("All final NMF outputs are present.")
else:
    print("Some final NMF outputs are missing.")